# 04 - BM25 Legal Information Retrieval

Implements a keyword-based BM25 baseline and evaluates three legal queries using Precision@5, Recall@5, and nDCG@5.

**Expected input:** `data/processed/preprocessed_wto_cases.csv`

> Install dependencies once from the repository root with `pip install -r requirements.txt`. Run the notebooks in numerical order.

In [ ]:
from pathlib import Path

# Resolve repository paths whether Jupyter starts in the repository root or notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_PDF_DIR = DATA_DIR / "raw" / "wto_case_pdfs"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

RAW_PDF_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_PATH = PROCESSED_DIR / "wto_cases.csv"
PROCESSED_DATA_PATH = PROCESSED_DIR / "preprocessed_wto_cases.csv"

print(f"Project root: {PROJECT_ROOT}")


In [1]:
import pandas as pd

# Load the cleaned and pre-processed dataset 
df = pd.read_csv(PROCESSED_DATA_PATH)

# Display the first few rows
df.head()

,Case ID,Title,Complainant,Respondent,Date,Articles,Summary,Title_normalized,Summary_normalized,Articles_normalized,Title_tokenized,Summary_tokenized,Articles_tokenized,Summary_lemmatized,Articles_lemmatized,Summary_segmented
0,ds491,US – COATED PAPER (INDONESIA)1,Indonesia,United States,22 January 2018,"ADA Arts. 3.5, 3.7, 3.8;\nASCM Arts. 2.1(c), 1...",FINDINGS2\n• ASCM Art. 14(d) (rejection of in-...,us coated paper indonesia1,findings2\n ascm art 14d rejection of incountr...,ada arts 35 37 38\nascm arts 21c 127 14d 155\n...,us coated paper indonesia1,findings2 ascm art 14d rejection incountry pri...,ada arts 35 37 38 ascm arts 21c 127 14d 155 15...,findings2 ascm art 14d rejection incountry pri...,ada arts 35 37 38 ascm art 21c 127 14d 155 157...,['findings2 ascm art 14d rejection incountry p...
1,ds33,US – WOOL SHIRTS AND BLOUSES1,India,United States,23 May 1997,ATC Arts. 6 and 2.4,/AB FINDINGS\n• ATC Art. 6 (transitional safeg...,us wool shirts and blouses1,ab findings\n atc art 6 transitional safeguard...,atc arts 6 and 24,us wool shirts blouses1,ab findings atc art 6 transitional safeguard m...,atc arts 6 24,ab finding atc art 6 transitional safeguard me...,atc arts 6 24,['ab finding atc art 6 transitional safeguard ...
2,ds403,PHILIPPINES – DISTILLED SPIRITS1,"European Union,\nUnited States",Philippines,20 January 2012,"GATT Art. III:2, first and\nsecond sentences",/AB FINDINGS\n• GATT Art. III:2 (national trea...,philippines distilled spirits1,ab findings\n gatt art iii2 national treatment...,gatt art iii2 first and\nsecond sentences,philippines distilled spirits1,ab findings gatt art iii2 national treatment t...,gatt art iii2 first second sentences,ab findings gatt art iii2 national treatment t...,gatt art iii2 first second sentence,['ab findings gatt art iii2 national treatment...
3,ds267A,US – UPLAND COTTON (ARTICLE 21.5 – BRAZIL)1,Brazil,United States,20 June 2008,"ASCM Arts.3, 5(c), 6.3(c), and item\n(j) of th...","/AB FINDINGS\n• AA Arts. 10.1 and 8, and ASCM ...",us upland cotton article 215 brazil1,ab findings\n aa arts 101 and 8 and ascm arts ...,ascm arts3 5c 63c and item\nj of the illustrat...,us upland cotton article 215 brazil1,ab findings aa arts 101 8 ascm arts 31a 32 ill...,ascm arts3 5c 63c item j illustrative list aa ...,ab findings aa arts 101 8 ascm art 31a 32 illu...,ascm arts3 5c 63c item j illustrative list aa ...,['ab findings aa arts 101 8 ascm art 31a 32 il...
4,ds44,JAPAN – FILM1,United States,Japan,22 April 1998,"GATT Arts. XXIII:1(b), III:4 and X:1",FINDINGS\n• GATT Art. XXIII:1(b) (non-violatio...,japan film1,findings\n gatt art xxiii1b nonviolation claim...,gatt arts xxiii1b iii4 and x1,japan film1,findings gatt art xxiii1b nonviolation claim p...,gatt arts xxiii1b iii4 x1,findings gatt art xxiii1b nonviolation claim p...,gatt arts xxiii1b iii4 x1,['findings gatt art xxiii1b nonviolation claim...


In [2]:
df['text_for_retrieval'] = (
    df['Title_tokenized'].fillna('') + ' ' +
    df['Articles_lemmatized'].fillna('') + ' ' +
    df['Summary_lemmatized'].fillna('')
)

In [4]:
from rank_bm25 import BM25Okapi
from nltk.tokenize import word_tokenize
import nltk
from sklearn.metrics import ndcg_score
import pandas as pd

nltk.download('punkt')

# Tokenize the corpus
tokenized_corpus = [word_tokenize(doc.lower()) for doc in df['text_for_retrieval']]

# Initialize BM25
bm25 = BM25Okapi(tokenized_corpus)

# Define the query
query = "US - China Case Laws using Article VIII of GATT"
tokenized_query = word_tokenize(query.lower())

# Compute BM25 scores
scores = bm25.get_scores(tokenized_query)

# Get top 5 indices
top_n = 5
top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_n]

# Retrieve top 5 cases (Case ID and Title only)
top_bm25_cases = df.iloc[top_indices][["Case ID", "Title"]]
bm25_results = top_bm25_cases["Case ID"].tolist()

# Display ranked results
print(" BM25 Top 5 Results:")
for rank, (_, row) in enumerate(top_bm25_cases.iterrows(), start=1):
    print(f"{rank}. {row['Case ID']} – {row['Title']}")

# ---------------------------------------------------
# 📊 Evaluation Metrics (Precision, Recall, nDCG@5)
# ---------------------------------------------------

# Ground truth
ground_truth = {"ds437", "ds440", "ds437A", "ds517", "ds395"}  

# Binary relevance vector (1 if in ground truth, else 0)
relevance_vector = [1 if case_id in ground_truth else 0 for case_id in bm25_results]

# Precision@5
precision_at_5 = sum(relevance_vector) / len(relevance_vector)

# Recall@5
recall_at_5 = sum(relevance_vector) / len(ground_truth) if ground_truth else 0.0

# nDCG@5 using inverse rank as score
y_true = [relevance_vector]                      # actual relevance
y_score = [[1 / (i + 1) for i in range(5)]]     # heuristic scores by rank

ndcg_at_5 = ndcg_score(y_true, y_score)

# Print evaluation metrics
print("\n BM25 Evaluation Metrics:")
print(f"Precision@5: {precision_at_5:.4f}")
print(f"Recall@5: {recall_at_5:.4f}")
print(f"nDCG@5: {ndcg_at_5:.4f}")


 BM25 Top 5 Results:
1. ds395 – CHINA – RAW MATERIALS1
2. ds422 – US – SHRIMP AND SAWBLADES (CHINA)1
3. ds432 – CHINA – RARE EARTHS1
4. ds339 – CHINA – AUTO PARTS1
5. ds437 – US – COUNTERVAILING MEASURES (CHINA)1

 BM25 Evaluation Metrics:
Precision@5: 0.4000
Recall@5: 0.4000
nDCG@5: 0.8503


[nltk_data] Downloading package punkt to
[nltk_data]     <LOCAL_NLTK_DATA>...
[nltk_data]   Package punkt is already up-to-date!


In [11]:
from rank_bm25 import BM25Okapi
from nltk.tokenize import word_tokenize
import nltk
from sklearn.metrics import ndcg_score
import pandas as pd

nltk.download('punkt')

# Tokenize the corpus
tokenized_corpus = [word_tokenize(doc.lower()) for doc in df['text_for_retrieval']]

# Initialize BM25
bm25 = BM25Okapi(tokenized_corpus)

# Define the query
query = "Case Laws involving European Communities as Complainant and United States as Respondent"
tokenized_query = word_tokenize(query.lower())

# Compute BM25 scores
scores = bm25.get_scores(tokenized_query)

# Get top 5 indices
top_n = 5
top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_n]

# Retrieve top 5 cases (ID and Title only)
top_bm25_cases = df.iloc[top_indices][["Case ID", "Title"]]
bm25_results = top_bm25_cases["Case ID"].tolist()

# Display ranked results
print(" BM25 Top 5 Results:")
for rank, (_, row) in enumerate(top_bm25_cases.iterrows(), start=1):
    print(f"{rank}. {row['Case ID']} – {row['Title']}")

# ---------------------------------------------------
# 📊 Evaluation Metrics (Precision, Recall, nDCG@5)
# ---------------------------------------------------

# Ground truth
ground_truth = {"ds353A", "ds108A", "ds108B", "ds165", "ds34"}  

# Binary relevance vector (1 if in ground truth, else 0)
relevance_vector = [1 if case_id in ground_truth else 0 for case_id in bm25_results]

# Precision@5
precision_at_5 = sum(relevance_vector) / len(relevance_vector)

# Recall@5
recall_at_5 = sum(relevance_vector) / len(ground_truth) if ground_truth else 0.0

# nDCG@5 using inverse rank as score
y_true = [relevance_vector]                      # actual relevance
y_score = [[1 / (i + 1) for i in range(5)]]     # heuristic scores by rank

ndcg_at_5 = ndcg_score(y_true, y_score)

# Print evaluation metrics
print("\n BM25 Evaluation Metrics:")
print(f"Precision@5: {precision_at_5:.4f}")
print(f"Recall@5: {recall_at_5:.4f}")
print(f"nDCG@5: {ndcg_at_5:.4f}")


 BM25 Top 5 Results:
1. ds266 – EC – EXPORT SUBSIDIES ON SUGAR1
2. ds165 – US – CERTAIN EC PRODUCTS1
3. ds34 – TURKEY – TEXTILES1
4. ds62 – EC – COMPUTER EQUIPMENT1
5. ds108B – US – FSC (ARTICLE 21.5 – EC II)1

 BM25 Evaluation Metrics:
Precision@5: 0.6000
Recall@5: 0.6000
nDCG@5: 0.7123


[nltk_data] Downloading package punkt to
[nltk_data]     <LOCAL_NLTK_DATA>...
[nltk_data]   Package punkt is already up-to-date!


In [12]:
from rank_bm25 import BM25Okapi
from nltk.tokenize import word_tokenize
import nltk
from sklearn.metrics import ndcg_score
import pandas as pd

nltk.download('punkt')

# Tokenize the corpus
tokenized_corpus = [word_tokenize(doc.lower()) for doc in df['text_for_retrieval']]

# Initialize BM25
bm25 = BM25Okapi(tokenized_corpus)

# Define the query
query = "Case Laws involving Australia with ASCM Article Application"
tokenized_query = word_tokenize(query.lower())

# Compute BM25 scores
scores = bm25.get_scores(tokenized_query)

# Get top 5 indices
top_n = 5
top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_n]

# Retrieve top 5 cases (ID and Title only)
top_bm25_cases = df.iloc[top_indices][["Case ID", "Title"]]
bm25_results = top_bm25_cases["Case ID"].tolist()

# Display ranked results
print(" BM25 Top 5 Results:")
for rank, (_, row) in enumerate(top_bm25_cases.iterrows(), start=1):
    print(f"{rank}. {row['Case ID']} – {row['Title']}")

# ---------------------------------------------------
# 📊 Evaluation Metrics (Precision, Recall, nDCG@5)
# ---------------------------------------------------

# Ground truth
ground_truth = {"ds126", "ds126A", "ds367", "ds18A", "ds529"}  

# Binary relevance vector (1 if in ground truth, else 0)
relevance_vector = [1 if case_id in ground_truth else 0 for case_id in bm25_results]

# Precision@5
precision_at_5 = sum(relevance_vector) / len(relevance_vector)

# Recall@5
recall_at_5 = sum(relevance_vector) / len(ground_truth) if ground_truth else 0.0

# nDCG@5 using inverse rank as score
y_true = [relevance_vector]                      # actual relevance
y_score = [[1 / (i + 1) for i in range(5)]]     # heuristic scores by rank

ndcg_at_5 = ndcg_score(y_true, y_score)

# Print evaluation metrics
print("\n BM25 Evaluation Metrics:")
print(f"Precision@5: {precision_at_5:.4f}")
print(f"Recall@5: {recall_at_5:.4f}")
print(f"nDCG@5: {ndcg_at_5:.4f}")


 BM25 Top 5 Results:
1. ds126A – AUSTRALIA – AUTOMOTIVE LEATHER II (ARTICLE 21.5 – US)1
2. ds126 – AUSTRALIA – AUTOMOTIVE LEATHER II1
3. ds18A – AUSTRALIA – SALMON (ARTICLE 21.5 – CANADA)1
4. ds353 – US – LARGE CIVIL AIRCRAFT (2ND COMPLAINT)1
5. ds367 – AUSTRALIA – APPLES1

 BM25 Evaluation Metrics:
Precision@5: 0.8000
Recall@5: 0.8000
nDCG@5: 0.9829


[nltk_data] Downloading package punkt to
[nltk_data]     <LOCAL_NLTK_DATA>...
[nltk_data]   Package punkt is already up-to-date!
